EA refund calculation - Download from CST 

[Preparation]
1. Usage download from CST - Invoice Usage csv file
2. sub_list csv file

**For sub_list.csv file you can filter by subId or subname.

1) Filter SubID: create csv file with first column input by 'S|ubID'
2) Filter SubName: create csv file with first column input by 'SubName'


In [1]:
import pandas as pd
import dask.dataframe as dd

pd.options.display.float_format = '{:.7f}'.format
pd.set_option('mode.chained_assignment', None)

#Filter by subName please Input "Y" after subName=
#Filter by subID please Input "N" after subName=
#no filter use please Input "S" after subName=
subName = "N"

##[Filter by Sub ID]
# Input your sublist file name in '' from read_csv('') including .csv file format 
# And delete "#" infront of subInfo= 
# Then add "#" infront of second subInfo= 
subInfo=pd.read_csv('sub_llist.csv')['SubID'].values.tolist() 

##[Filter by Sub Name]
# Input your sublist file name in '' from read_csv('') including .csv file format 
# And delete "#" infront of subInfo= 
# Then add "#" infront of first subInfo= 
#subInfo=pd.read_csv('mysubName.csv')['name'].values.tolist() 

# Input your usage download file name after url= including .csv file format
url = 'Detail_Enrollment_61419529__en.csv' 

usage = dd.read_csv(url, usecols=['SubscriptionId', 'SubscriptionName','PricingModel','ChargeType','Cost','Date','ReservationId','ReservationName', 'Product', 'benefitId']
                    ,low_memory=False
                   ,dtype={'ReservationId': 'object',
                           'ReservationName': 'object',
                           'benefitId': 'object'
                          })
subList = []

if(subName == "N"):
    for a in subInfo:
        if a not in subList:
            subList.append(a)
    filtered = usage.loc[usage["SubscriptionId"].isin(subList)]
    pd_usage = filtered.compute()
elif (subName == "Y"):
    for a in subInfo:
        if a not in subList:
            subList.append(a)
    filtered = usage.loc[usage["SubscriptionName"].isin(subList)]
    pd_usage = filtered.compute()
else:
    pd_usage = usage.compute()

cal_usage = pd_usage.fillna('NaN')
print("Usage Filtering Done!!! Please proceed cell below!")

Usage Filtering Done!!! Please proceed cell below!


In [14]:
####Total Usage######################################################################################################
summary = pd.pivot_table(cal_usage, index=["SubscriptionId", "SubscriptionName", "Product"], values=["Cost"], aggfunc="sum", margins=True, margins_name="Total")
result = pd.DataFrame(summary.to_records())
display(summary)

,,,Cost
SubscriptionId,SubscriptionName,Product,
Total,,,0.0000000


In [15]:
####Calculate Usage refund###########################################################################################
TotalAmount = 0
refund = 0
usageFlag = True

if(cal_usage.empty):
    print("#### No Usage ####")
    usageFlag = False
else:
    usage = pd.pivot_table(cal_usage, index=["SubscriptionId"], values=["Cost"], aggfunc="sum", margins=True, margins_name="Total")
    usage.reset_index(inplace=True)
    TotalAmount = usage.tail(n=1)['Cost']
    refund = TotalAmount*0.1
    usage.loc[len(usage.index)] = ['Refund(Total*10%)',refund.values[0]]
    display(usage)

#### No Usage ####


In [16]:
##Download
file_name = 'refundResult.xlsx'
with pd.ExcelWriter(file_name) as writer:
#    total.to_excel(writer, sheet_name="Total Refund Amount", startrow=1, startcol=1)
    if(usageFlag):
        usage.to_excel(writer, sheet_name="Usage Refund Calc", startrow=1, startcol=1, index=False)
#    if(riFlag):
#        riTable.to_excel(writer, sheet_name="RI Refund Calc", startrow=1, startcol=1, index=False)
    cal_usage.to_excel(writer, sheet_name="raw Usage Summary", startrow=1, startcol=1, index=False)
display("DONE!! " + file_name + " file")

'DONE!! refundResult.xlsx file'